In [1]:
from epo.tipdata.patstat import PatstatClient
import pandas as pd
import time

# Connect to PATSTAT
patstat = PatstatClient(env='PROD')

def timed_query(query):
    """Execute query and return DataFrame with timing."""
    start = time.time()
    res = patstat.sql_query(query, use_legacy_sql=False)
    print(f"Query took {time.time() - start:.2f}s ({len(res)} rows)")
    return pd.DataFrame(res)

# Run the query
df = timed_query("""
WITH ai_erp_patents AS (
                SELECT DISTINCT
                    a.appln_id,
                    a.appln_filing_date,
                    a.appln_filing_year
                FROM tls201_appln a
                WHERE a.appln_filing_year BETWEEN 2014 AND 2023
                  AND a.appln_auth IN ('EP', 'DE')
                  AND EXISTS (
                    SELECT 1 FROM tls224_appln_cpc cpc1
                    WHERE cpc1.appln_id = a.appln_id
                      AND cpc1.cpc_class_symbol LIKE 'G06Q%'
                      AND cpc1.cpc_class_symbol LIKE '%10/%'
                  )
                  AND EXISTS (
                    SELECT 1 FROM tls224_appln_cpc cpc2
                    WHERE cpc2.appln_id = a.appln_id
                      AND cpc2.cpc_class_symbol LIKE 'G06N%'
                  )
            ),
            top_applicants AS (
                SELECT
                    p.person_id,
                    p.person_name,
                    p.person_ctry_code,
                    p.psn_sector,
                    COUNT(DISTINCT aep.appln_id) AS patent_count,
                    MIN(aep.appln_filing_date) AS first_filing_date,
                    MAX(aep.appln_filing_date) AS latest_filing_date,
                    COUNT(DISTINCT aep.appln_filing_year) AS active_years
                FROM ai_erp_patents aep
                JOIN tls207_pers_appln pa ON aep.appln_id = pa.appln_id
                JOIN tls206_person p ON pa.person_id = p.person_id
                WHERE pa.applt_seq_nr > 0
                GROUP BY p.person_id, p.person_name, p.person_ctry_code, p.psn_sector
                ORDER BY patent_count DESC
                LIMIT 10
            )
            SELECT
                person_name,
                person_ctry_code AS country,
                psn_sector AS sector,
                patent_count,
                active_years,
                CAST(first_filing_date AS STRING) AS first_filing,
                CAST(latest_filing_date AS STRING) AS latest_filing
            FROM top_applicants
            ORDER BY patent_count DESC
""")

# Display results
df

Query took 4.28s (10 rows)


,person_name,country,sector,patent_count,active_years,first_filing,latest_filing
0,Siemens Aktiengesellschaft,DE,COMPANY,83,9,2015-06-29,2023-12-20
1,"Microsoft Technology Licensing, LLC",US,COMPANY,76,9,2015-03-13,2023-08-22
2,Tata Consultancy Services Limited,IN,COMPANY,64,9,2014-09-05,2023-10-20
3,Accenture Global Solutions Limited,IE,COMPANY,37,6,2017-01-27,2022-07-05
4,Google LLC,US,COMPANY,35,7,2016-02-16,2022-10-12
5,Robert Bosch Gesellschaft mit beschränkter Haf...,DE,COMPANY,34,4,2020-03-06,2023-12-07
6,"Microsoft Technology Licensing, LLC",US,COMPANY,33,4,2016-04-27,2019-11-05
7,"Hitachi, Ltd.",JP,COMPANY,32,8,2014-03-25,2023-06-06
8,GENERAL ELECTRIC COMPANY,US,COMPANY,30,9,2015-12-09,2023-06-22
9,Geoquest Systems B.V.,NL,COMPANY,29,6,2018-08-21,2023-03-24
